# control

> A local socket a detached job runner listens on, so a host's page can prompt, steer, stop and answer its agent from another process.

In [ ]:
#| default_exp control

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import json, os, socket, socketserver, stat, tempfile, threading, time
from dataclasses import asdict
from pathlib import Path
from shalya import acts, err
from ramabana.core import agent_err
from ramabana.agent import TurnResult, model_title

## The socket

A program that hosts agent jobs, such as a web app, often runs each job in a process of its own. Restarting the web server then does not kill a turn halfway through. The page that started the job needs a way to reach it: to send a prompt, steer the running turn, stop it, and answer the approval requests it raises. `Control` provides that way. It opens a Unix socket in the session's run folder. A Unix socket is an address in the file system that only processes on this machine can connect to. Only the current user may connect.

The protocol is one JSON object per line in each direction. A request is `{"id", "method", "params"}`. A successful reply is `{"id", "result"}`. A failed reply is `{"id", "error"}`. `METHODS` lists the methods:

- `prompt(text)` starts a turn on a worker thread. If a turn is already running, `text` is queued to run after it.
- `steer(text)` gives `text` to the running turn with its next tool result. This redirects the model without stopping the turn.
- `cancel()` stops the running turn and drops the prompts queued behind it. The reply includes the number dropped as `dropped`.
- `asks()` lists the approval requests waiting for an answer, oldest first. `answer(id, ok, note)` answers one of them.
- `status()` returns `running`, `queued` (the number of queued prompts), `session` and `last`. `last` is how the last turn ended, or `None` before any turn has ended. It also returns what a peer lists: `name`, the session's name for a person; `state`, one of `idle`, `working` and `waiting for approval`; `folder`, its first open folder; and `at`, when it was last active.
- `events(offset)` returns `events`, the records in the session's event file from byte `offset` on, and `offset`, the value to pass next time. It returns whole lines only, at most `EVENTS_MAX` bytes per call. When the file has rolled over since the last read, it starts from byte 0 and sets `rotated` to true. It raises an error until the runner has called `agent.events_to(path)`.

`control_call(path, method, **params)` is the other end. It connects, sends one request and returns the `result`. On an error reply it raises `RuntimeError` with the error's message. Its `timeout` argument is 30 seconds by default. The page uses it, and any script that needs to reach a running job can use it.

`start` refuses a path where another runner is listening and a path that holds anything other than a socket. It removes a socket that a crashed runner left behind.

A Unix socket path takes at most about 100 bytes (`SOCKET_MAX`). When you give no path and the session's run folder is deeper, the socket goes in the temporary folder as `ramabana-<session>.sock`. An explicit `path` that is too long raises `ValueError`.

| Constant | Value | Limit |
|---|---|---|
| `SOCKET_MAX` | 100 | Bytes of a socket path. macOS allows 104. |
| `REQUEST_MAX` | 1,000,000 | Bytes of one request line |
| `EVENTS_MAX` | 1,000,000 | Bytes of the event file that one `events` call returns |

In [ ]:
#| export
CONTROL = 'control.sock'   #: in a session's run folder: the socket `Control` listens on
METHODS = ('prompt', 'steer', 'cancel', 'asks', 'answer', 'status', 'events')
SOCKET_MAX = 100   #: bytes a Unix socket path may take; macOS allows 104
REQUEST_MAX = 1_000_000   #: bytes one request line may take
EVENTS_MAX = 1_000_000    #: bytes of the event file one `events` call returns
EVENT_FILE = 'events.jsonl'   #: in a session's run folder: the event file a terminal session writes when nothing else does

def control_path(agent):
    "Where `agent`'s session listens: `CONTROL` in its run folder, or `ramabana-<session>.sock` in the temporary folder when that path is too long for a socket."
    p = Path(agent.runs_dir)/CONTROL
    return p if len(str(p).encode()) <= SOCKET_MAX else Path(tempfile.gettempdir())/f'ramabana-{agent.session_id}.sock'

def ago(t):
    "How long ago `t` was: `5m`, `3h` or `2d`."
    s = max(0, time.time() - (t or 0))
    return f'{int(s // 60)}m' if s < 3600 else f'{int(s // 3600)}h' if s < 86400 else f'{int(s // 86400)}d'

def pick_row(rows, choice):
    "The row numbered `choice` in `rows` as they were listed from 1, or None when `choice` is not one of those numbers."
    return rows[int(choice) - 1] if str(choice).isdigit() and 1 <= int(choice) <= len(rows) else None

def session_name(agent, at):
    "A live session's name for a person, titled as `Agent.sessions` titles it for `/resume`, else by its first folder's name, then its model and how long ago it was `at`."
    m = agent.session_meta()   # the stored index row: `sessions()` would rebuild the whole index on every `status`
    title = (m['title'] if m.get('manual') else model_title(m['title'])) or m.get('first_title', '')
    roots = list(agent.host.roots)
    return f"{title or (Path(roots[0]).name or roots[0] if roots else agent.session_id)} · {agent.model.name} · {ago(at)} ago"

class Control:
    "A Unix socket through which another process prompts, steers, stops and answers an agent running in this one."
    def __init__(self, agent,
                 path=None):   # the socket; None puts it in the session's run folder as `CONTROL`
        self.agent, self.last = agent, None
        self.path = Path(path) if path else control_path(agent)
        if len(str(self.path).encode()) > SOCKET_MAX: raise ValueError(f'a Unix socket path is at most {SOCKET_MAX} bytes: {self.path}')
        self.at = time.time()   #: when this session last started or finished a turn
        self._lock, self._running, self._server, self._inode = threading.Lock(), False, None, None

    def start(self):
        "Listen on `path` for this user only and return `self`. Raises `RuntimeError` when another runner listens on `path` or when `path` is not a socket."
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if self.path.exists() or self.path.is_symlink():
            if not stat.S_ISSOCK(self.path.lstat().st_mode): raise RuntimeError(f'{self.path} is not a socket; nothing was removed')
            with socket.socket(socket.AF_UNIX) as probe:
                try: probe.connect(str(self.path))
                except OSError: self.path.unlink()   # a runner that crashed left it
                else: raise RuntimeError(f'another runner listens on {self.path}')
        ctl = self
        class Handler(socketserver.StreamRequestHandler):
            def handle(self):
                while (raw := self.rfile.readline(REQUEST_MAX)):
                    try: req = json.loads(raw)
                    except ValueError as e: req = e
                    out = ctl.handle(req) if isinstance(req, dict) else {'id': None, 'error': {'message': f'not a JSON request object: {req}'}}
                    self.wfile.write((json.dumps(out, default=str) + '\n').encode())
                    self.wfile.flush()
        self._server = socketserver.ThreadingUnixStreamServer(str(self.path), Handler)
        os.chmod(self.path, 0o600)   # connecting needs write access; the default umask already keeps it from others
        self._inode = self.path.stat().st_ino
        self._server.daemon_threads = True
        threading.Thread(target=self._server.serve_forever, daemon=True, name='ramabana-control').start()
        return self

    def close(self):
        "Stop listening and remove the socket, unless another runner has replaced it."
        if self._server is not None: self._server.shutdown(); self._server.server_close(); self._server = None
        try:
            if self._inode is not None and self.path.stat().st_ino == self._inode: self.path.unlink()
        except OSError: pass
        self._inode = None

    def handle(self, req):
        "One request as a dict; its reply as `{'id', 'result'}` or `{'id', 'error'}`."
        rid, name, params = req.get('id'), req.get('method', ''), req.get('params') or {}
        if name not in METHODS: return {'id': rid, 'error': {'message': f'no method {name!r}; one of {", ".join(METHODS)}'}}
        try: return {'id': rid, 'result': getattr(self, f'do_{name}')(**params)}
        except Exception as e: return {'id': rid, 'error': {'message': agent_err(e)}}

    def _work(self, text):
        "Run `text`, then each prompt queued meanwhile. The last queue check and the end of running happen under one lock."
        try:
            while text is not None:
                try: self.last = self.agent.ask(text, result=True)
                except Exception as e: self.last = TurnResult(state='failed', error=agent_err(e), session=self.agent.session_id)
                self.at = time.time()
                with self._lock:
                    text = self.agent.queued.popleft() if self.agent.queued else None
                    if text is None: self._running = False
        except BaseException:   # only an error that escaped: a normal end already cleared it, and a new turn may have set it since
            with self._lock: self._running = False
            raise

    def do_prompt(self, text):
        "Start a turn on `text`, or queue it behind the one running."
        with self._lock:
            if self._running: return {'queued': self.agent.queue(text)}
            self._running = True
        self.at = time.time()
        threading.Thread(target=self._work, args=(text,), daemon=True, name='ramabana-control-turn').start()
        return {'started': True}

    def do_steer(self, text): return {'told': self.agent.steer(text)}
    def do_cancel(self):
        "Stop the running turn, and drop the prompts queued behind it."
        with self._lock:
            dropped = len(self.agent.queued)
            self.agent.queued.clear()
        return self.agent.cancel() | {'dropped': dropped}
    def do_asks(self): return [a.dict() for a in self.agent.approvals.waiting] if self.agent.approvals is not None else []

    def do_answer(self, id, ok, note=''):
        ap = self.agent.approvals
        return ap is not None and ap.answer(id, bool(ok), note) is not None

    def busy(self):
        "Whether a turn runs, and how many prompts wait behind it."
        return self._running, len(self.agent.queued)

    def active_at(self):
        "When this session was last active."
        return self.at

    def do_status(self):
        running, queued = self.busy()
        roots, at, ap = list(self.agent.host.roots), self.active_at(), self.agent.approvals
        return {'running': running, 'queued': queued, 'session': self.agent.session_id,
                'last': asdict(self.last) if self.last is not None else None,
                'name': session_name(self.agent, at), 'folder': roots[0] if roots else '', 'at': at,
                'state': 'waiting for approval' if ap is not None and ap.waiting else 'working' if running else 'idle'}

    def do_events(self, offset=0):
        files = self.agent.__dict__.get('_event_files') or ()
        if not files: raise RuntimeError('no event file; the runner calls agent.events_to(path) first')
        p = files[0][0]
        rotated = offset > p.stat().st_size   # the file rolled over to `.1` since the page last read it
        if rotated: offset = 0
        with open(p, 'rb') as f:
            f.seek(offset)
            data = f.read(EVENTS_MAX)
        cut = data.rfind(b'\n') + 1   # whole lines only: one still being written waits for the next read
        return {'events': [json.loads(l) for l in data[:cut].splitlines() if l.strip()], 'offset': offset + cut, 'rotated': rotated}


def control_call(path, method, timeout=30, **params):
    "Send one request to the `Control` socket at `path` and return its `result`; an error reply raises `RuntimeError`."
    with socket.socket(socket.AF_UNIX) as s:
        s.settimeout(timeout)
        s.connect(str(path))
        s.sendall((json.dumps({'id': 1, 'method': method, 'params': params}) + '\n').encode())
        out = json.loads(s.makefile('rb').readline())
    if 'error' in out: raise RuntimeError(out['error']['message'])
    return out['result']

The example plays both sides in one process. `fake_agent` from `ramabana.testing` supplies an agent whose model replies with a scripted line. The first cell sends a prompt through the socket, waits for the turn to finish, checks how it ended and reads the turn's events.

The second cell raises an approval request for `edit_file` from another thread, as a tool would, finds it through `asks`, and approves it through the socket. The agent was built with `no_listener='wait'`. With that setting the request waits for an answer. Without it, the request is refused because no terminal is attached. An unknown method comes back as an error, and `control_call` raises it as `RuntimeError`.

In [ ]:
import tempfile, time
from ramabana import agent
from ramabana.testing import fake_agent

d = Path(tempfile.mkdtemp(dir='/tmp'))   # a Unix socket path must stay short
a, be = fake_agent(cfg=d, replies=['the first reply'], approvals=agent.Approvals(tools={'edit_file'}, timeout=5, no_listener='wait'))
a.events_to(d/'job.events')
ctl = Control(a).start()
call = lambda method, **kw: control_call(ctl.path, method, **kw)

test_eq(call('prompt', text='hello'), {'started': True})
for _ in range(200):
    if not call('status')['running']: break
    time.sleep(.01)
st = call('status')
test_eq((st['running'], st['last']['state'], st['last']['reply']), (False, 'complete', 'the first reply'))
ev = call('events')
assert {'turn_start', 'turn_end'} <= {e['kind'] for e in ev['events']} and ev['offset'] > 0
test_eq(call('events', offset=ev['offset'])['events'], [])

In [ ]:
out = {}
threading.Thread(target=lambda: out.setdefault('r', a.approvals.request('edit_file', {'path': 'a.py'})), daemon=True).start()
for _ in range(200):
    if (waiting := call('asks')): break
    time.sleep(.01)
test_eq(waiting[0]['tool'], 'edit_file')
assert call('answer', id=waiting[0]['id'], ok=True)
for _ in range(200):
    if 'r' in out: break
    time.sleep(.01)
assert out['r'], 'answered through the socket'
test_fail(lambda: call('fly'), exc=RuntimeError)
ctl.close(); a.close()

## Peers

Every terminal session listens on its own socket, so one session can reach another: it can steer the other's running turn, or prompt it, and read the reply from its events. `peers_of(agent)` finds the other live sessions under the same config folder. It asks each socket in `<cfg>/runs/*/` and each `ramabana-<session>.sock` of a session in that folder for its `status`, and leaves out a socket that does not answer within `PEER_WAIT` seconds. The list puts the most recently active session first.

`status` names a session for a person, the way the `--resume` picker lists sessions: its title, else its first folder's name, then its model and how long ago it was active, as in `orgs · claude-opus-5-5 · 2m ago`. `peer_lines` numbers the list, and `find_peer` takes a number from it, as the picker takes one, or a session id.

`peer_tools(agent)` gives the agent the same reach as two tools. `peers()` lists the sessions with their ids. `tell(session, text, how='steer')` steers a session, or with `how='prompt'` starts or queues a turn there, and waits up to `wait` seconds for the reply of the turn that reads the text. It finds that turn in the session's events: the turn that started on the text, or the turn that was steered with it. A prompt into another session acts there, so `tell` asks the approval gate first, as a write does.

In [ ]:
#| export
PEER_WAIT = 2   #: seconds a peer has to answer `status` before `peers_of` leaves it out

def peers_of(agent, timeout=PEER_WAIT):
    "The other live sessions under `agent`'s config folder, most recently active first: each one's `status` plus the `path` it listens on."
    if agent.cfg is None: return []
    runs = Path(agent.cfg)/'runs'
    tmp = (p for p in Path(tempfile.gettempdir()).glob('ramabana-*.sock') if (runs/p.stem.removeprefix('ramabana-')).is_dir())
    out = []
    for p in [*runs.glob(f'*/{CONTROL}'), *tmp]:
        try: st = control_call(p, 'status', timeout=timeout)
        except (OSError, RuntimeError, ValueError): continue   # a crashed runner's socket, or one too busy to answer
        if st.get('session') != agent.session_id: out.append(st | {'path': str(p)})
    return sorted(out, key=lambda s: s.get('at') or 0, reverse=True)

def peer_lines(rows, ids=False):
    "`rows` numbered from 1, one line each: name, state and folder, and with `ids` the session id."
    return '\n'.join(f"{i}  {s.get('name') or s['session']}  {s.get('state', '')}  {s.get('folder', '')}" + (f"  ({s['session']})" if ids else '')
                     for i, s in enumerate(rows, 1))

def peer_label(row):
    "A peer's name without how long ago it was active, which a label kept on screen would leave stale."
    return (row.get('name') or row['session']).rsplit(' · ', 1)[0]

def find_peer(rows, key):
    "The row `key` names: its number in `rows`, or its session id; None when neither matches."
    return pick_row(rows, key) or next((s for s in rows if s['session'] == str(key)), None)

def events_end(path):
    "The offset just past the last whole event the session at `path` has written."
    off = 0
    while (nxt := control_call(path, 'events', offset=off)['offset']) != off: off = nxt
    return off

def carries(said, text):
    "Whether `said`, a turn's prompt or a steered message, holds `text` whole: as all of it, one paragraph or one line. An event clips a prompt at 2000 characters, so a longer `text` matches on its start."
    t, said = text.strip(), said.strip()
    if len(t) > 1000: return t[:1000] in said
    return t == said or t in {p.strip() for p in (*said.split('\n\n'), *said.splitlines())}

def await_reply(path, offset, text, timeout):
    "The `turn_end` event of the turn that reads `text`, from the events of the session at `path` after `offset`; None when it has not ended within `timeout` seconds."
    end, turn = time.monotonic() + timeout, None
    while True:
        got = control_call(path, 'events', offset=offset)
        offset = got['offset']
        for e in got['events']:
            if turn is None and ((e['kind'] == 'turn_start' and carries(e.get('prompt', ''), text))
                                 or (e['kind'] == 'steer' and any(carries(m, text) for m in e.get('messages', ())))): turn = e.get('turn_id')
            elif turn is not None and e['kind'] == 'turn_end' and e.get('turn_id') == turn: return e
        if time.monotonic() >= end: return None
        time.sleep(.2)

def peer_tools(agent):
    "`peers()` and `tell()`: the agent's tools for reaching the other live sessions."
    shown = []
    def peers() -> str:
        "List the other live Ramabana sessions, numbered, most recently active first: name, state, folder and session id."
        shown[:] = peers_of(agent)
        return peer_lines(shown, ids=True) or 'no other session is running'
    @acts
    def tell(
        session: str,         # a number from the last `peers()` list, or a session id
        text: str,            # what to say
        how: str = 'steer',   # `steer` the running turn there, or `prompt` to start or queue a turn there
        wait: int = 60,       # seconds to wait for the reply of the turn that reads it; 0 returns at once
    ) -> str:
        "Send `text` to another live session and return the reply of the turn that reads it. A prompt asks the user first."
        if how not in ('steer', 'prompt'): return err(f"how is 'steer' or 'prompt', not {how!r}")
        if not text.strip(): return err('nothing to say')
        if (row := find_peer(shown, session)) is None:
            shown[:] = peers_of(agent)
            row = find_peer(shown, session)
        if row is None: return err(f'no live session {session!r}; peers() lists them')
        name, path = peer_label(row), row['path']
        if how == 'prompt' and agent.approvals is not None:
            ask = agent.approvals.request('tell', {'session': name, 'how': how, 'text': text}, force=True)
            if not ask.answer: return err(ask.reply())
        try:
            try: start = events_end(path) if wait else None
            except RuntimeError: start = None   # a session with no event file still takes the text
            out = control_call(path, how, text=text)
            if how == 'steer' and not out.get('told'): return err(f"{name} is idle, so nothing read it; how='prompt' starts a turn there")
            said = f'told {name}' if how == 'steer' else f"{'started a turn in' if out.get('started') else 'queued it in'} {name}"
            if not wait: return said
            if start is None: return f'{said}; it writes no events, so its reply cannot be read from here'
            if (e := await_reply(path, start, text, wait)) is None: return f'{said}; no reply within {wait} s'
            return f"{said}; its reply ({e.get('state')}):\n{e.get('reply') or e.get('error') or ''}"
        except (OSError, RuntimeError) as e: return err(f'{name}: {agent_err(e)}')
    return [peers, tell]

Two more agents in the same config folder show the peer side. `listener` writes its events to `EVENT_FILE`, as a terminal session does, so a reply can be read from them. `asker` lists the sessions it can reach: the one that listens, by its folder's name since it has no title yet, and never itself.

In [ ]:
listener, _ = fake_agent(cfg=d, replies=['nothing much'])
asker, _ = fake_agent(cfg=d)
listener.events_to(listener.runs_dir/EVENT_FILE)
listening = Control(listener).start()
peers, tell = peer_tools(asker)
listed = peers()
assert listed.startswith('1  proj · fake · 0m ago  idle') and listener.session_id in listed and asker.session_id not in listed
listed

A steer reaches only a running turn, so `tell` says when nobody read it. A prompt starts a turn there, and `tell` returns that turn's reply. `asker` has no approval gate, so nothing asks first.

In [ ]:
assert 'idle' in tell(1, 'are you there?')
replied = tell(1, 'what are you working on?', how='prompt', wait=10)
listening.close(); listener.close(); asker.close()
assert replied.endswith('nothing much')
replied

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()